In [ ]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pytorch_lightning as pl
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import torchmetrics
import torchvision
import torchvision.transforms as T
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
from torch_uncertainty.methods import SWAG
from torch_uncertainty.models.classification.lenet import lenet

%matplotlib inline
sns.set_theme(style="whitegrid")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CIFAR-10 vs. SVHN

## Загрузка датасетов

In [ ]:
cifar_transforms = T.Compose(
    [
        T.ToTensor(),
        T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)

cifar_train = torchvision.datasets.CIFAR10(
    root="../data", train=True, download=True, transform=cifar_transforms
)
cifar_train_loader = DataLoader(
    cifar_train, batch_size=128, shuffle=True, num_workers=2
)

cifar_test = torchvision.datasets.CIFAR10(
    root="../data", train=False, download=True, transform=cifar_transforms
)
cifar_loader = DataLoader(
    cifar_test, batch_size=128, shuffle=False, num_workers=2
)

svhn_test = torchvision.datasets.SVHN(
    root="../data", split="test", download=True, transform=cifar_transforms
)
svhn_loader = DataLoader(
    svhn_test, batch_size=128, shuffle=False, num_workers=2
)

## Сбор SWAG статистики

In [ ]:
class SWAGFixed(SWAG):
    @torch.no_grad()
    def update_wrapper(self, epoch: int) -> None:
        """Update the SWAG posterior.

        The update is performed if the epoch is greater than the cycle start
        and the difference between the epoch and the cycle start is a multiple
        of the cycle length.

        Args:
            epoch : Current epoch.
        """
        if not (
            epoch > self.cycle_start
            and (epoch - self.cycle_start) % self.cycle_length == 0
        ):
            return

        # Эту штуку нужно использовать на cpu, но в оригинальном коде
        # она туда не переносится и все падает, а тестов для SWAG в основном
        # репо torch-uncertainty нет :(
        n_models = self.num_avgd_models.item()

        for name_p, param in self.core_model.named_parameters():
            mean = self.swag_stats[self.prfx + name_p + "_mean"]
            squared_mean = self.swag_stats[self.prfx + name_p + "_sq_mean"]
            new_param = param.data.detach().cpu()

            # Используем в этих двух формулах
            mean = mean * n_models / (n_models + 1) + new_param / (n_models + 1)
            squared_mean = squared_mean * n_models / (
                n_models + 1
            ) + new_param**2 / (n_models + 1)

            self.swag_stats[self.prfx + name_p + "_mean"] = mean
            self.swag_stats[self.prfx + name_p + "_sq_mean"] = squared_mean

            if not self.diag_covariance:
                covariance_sqrt = self.swag_stats[
                    self.prfx + name_p + "_covariance_sqrt"
                ]
                dev = (new_param - mean).view(-1, 1).t()
                covariance_sqrt = torch.cat((covariance_sqrt, dev), dim=0)
                if self.num_avgd_models + 1 > self.max_num_models:
                    covariance_sqrt = covariance_sqrt[1:, :]
                self.swag_stats[self.prfx + name_p + "_covariance_sqrt"] = (
                    covariance_sqrt
                )

        self.num_avgd_models += 1

        self.samples = [
            self.sample(self.scale, self.diag_covariance)
            for _ in range(self.num_estimators)
        ]
        self.need_bn_update = True
        self.fit = True

    def bn_update(
        self, loader: DataLoader, device: torch.device | str | int | None
    ) -> None:
        """Update the batchnorm statistics of the current SWAG samples.

        Args:
            loader: DataLoader to update the batchnorm statistics.
            device: Device to perform the update.
        """
        if self.need_bn_update:
            for mod in self.samples:
                mod.to(device)
                torch.optim.swa_utils.update_bn(loader, mod, device=device)
                mod.to("cpu")

                torch.cuda.empty_cache()

            self.need_bn_update = False


base_model = torch.hub.load(
    "chenyaofo/pytorch-cifar-models", "cifar10_vgg16_bn", pretrained=True
)
swag_model = SWAGFixed(
    base_model,
    cycle_start=1,
    cycle_length=2,
    num_estimators=20,
    max_num_models=10,
).to(device)

In [ ]:
optimizer = optim.SGD(
    swag_model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4
)
criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    swag_model.train()

    for inputs, labels in cifar_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = swag_model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

    swag_model.update_wrapper(epoch)

    print(f"Эпоха {epoch + 1}")

swag_model.bn_update(cifar_train_loader, device)
swag_model.eval();

## Получение неопределенности и визуализация

In [ ]:
@torch.no_grad()
def get_uncertainties(model, dataloader, num_passes):
    total, aleatoric, epistemic = [], [], []

    for images, _ in dataloader:
        images = images.to(device)
        output = model.eval_forward(images).view(num_passes, images.size(0), -1)

        probs = torch.softmax(output, dim=-1)
        p_mean = probs.mean(dim=0)

        total_uncertainty = -torch.sum(
            p_mean * torch.log(p_mean + 1e-12), dim=-1
        )
        aleatoric_uncertainty = -torch.sum(
            probs * torch.log(probs + 1e-12), dim=-1
        ).mean(dim=0)
        epistemic_uncertainty = total_uncertainty - aleatoric_uncertainty

        total.append(total_uncertainty.cpu().numpy())
        aleatoric.append(aleatoric_uncertainty.cpu().numpy())
        epistemic.append(epistemic_uncertainty.cpu().numpy())

    return (
        np.concatenate(total),
        np.concatenate(aleatoric),
        np.concatenate(epistemic),
    )


total_id, aleatoric_id, epistemic_id = get_uncertainties(
    swag_model, cifar_loader, 20
)
total_ood, aleatoric_ood, epistemic_ood = get_uncertainties(
    swag_model, svhn_loader, 20
)

In [ ]:
y_true = np.concatenate(
    [np.ones_like(epistemic_ood), np.zeros_like(epistemic_id)]
)

auroc_total = roc_auc_score(y_true, np.concatenate([total_ood, total_id]))

plt.figure(figsize=(6, 5))

sns.kdeplot(
    total_id, fill=True, color="#3498db", label="CIFAR-10 (ID)", clip=(0, None)
)
sns.kdeplot(
    total_ood, fill=True, color="#e74c3c", label="SVHN (OOD)", clip=(0, None)
)

plt.title(f"Полная неопределенность\nAUROC = {auroc_total:.3f}", fontsize=14)
plt.xlabel("Значение энтропии", fontsize=12)
plt.ylabel("Плотность", fontsize=12)
plt.legend()

plt.tight_layout()
plt.show()

SWAG, в отличие от MC Dropout, имеет более пологое распределение для OOD изображений с пиком в точке 1.2-1.3, а также лучший AUROC. Посмотрим на соотношение типов неопределенности.

In [ ]:
plt.figure(figsize=(6, 5))

plt.scatter(
    aleatoric_id,
    epistemic_id,
    alpha=0.3,
    color="#3498db",
    label="CIFAR-10 (ID)",
    s=15,
)
plt.scatter(
    aleatoric_ood,
    epistemic_ood,
    alpha=0.3,
    color="#e74c3c",
    label="SVHN (OOD)",
    s=15,
)

max_val = max(np.max(aleatoric_ood), np.max(epistemic_ood))
plt.plot(
    [0, max_val],
    [0, max_val],
    "k--",
    alpha=0.5,
    label="Эпистемическая = Алеаторной",
)

plt.title("Соотношение типов неопределенности", fontsize=14)
plt.xlabel("Алеаторная", fontsize=13)
plt.ylabel("Эпистемическая", fontsize=13)
plt.legend(fontsize=12)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Видно, что картина у SWAG намного лучше: для OOD изображений, ожидаемо, преобладающим типом неопределенности является эпистемическая.

# MNIST vs. Fashion-MNIST

## Загрузка датасетов

In [ ]:
mnist_transforms = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])

mnist_train = torchvision.datasets.MNIST(
    root="../data", train=True, download=True, transform=mnist_transforms
)
mnist_train_loader = DataLoader(
    mnist_train, batch_size=128, shuffle=True, num_workers=2
)

mnist_test = torchvision.datasets.MNIST(
    root="../data", train=False, download=True, transform=mnist_transforms
)
mnist_id_loader = DataLoader(
    mnist_test, batch_size=128, shuffle=False, num_workers=2
)

fmnist_test = torchvision.datasets.FashionMNIST(
    root="../data", train=False, download=True, transform=mnist_transforms
)
fmnist_ood_loader = DataLoader(
    fmnist_test, batch_size=128, shuffle=False, num_workers=2
)

## Обучение простой модели

In [ ]:
warnings.filterwarnings("ignore")

logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)


class LitLeNet(pl.LightningModule):
    def __init__(self):
        super().__init__()

        self.model = lenet(in_channels=1, num_classes=10)
        self.acc_metric = torchmetrics.Accuracy(
            task="multiclass", num_classes=10
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch):
        x, y = batch

        logits = self(x)
        loss = self.loss_fn(logits, y)
        acc = self.acc_metric(logits, y)

        self.log("train_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
        self.log("train_loss", loss, prog_bar=True)

        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)


lenet_model = LitLeNet().to(device)

trainer = pl.Trainer(
    max_epochs=1,
    accelerator="auto",
    logger=False,
    enable_checkpointing=False,
    enable_model_summary=False,
    enable_progress_bar=True,
)
trainer.fit(lenet_model, mnist_train_loader)

## Сбор SWAG статистики

In [ ]:
swag_model = SWAGFixed(
    lenet_model,
    cycle_start=1,
    cycle_length=2,
    num_estimators=20,
    max_num_models=10,
).to(device)

optimizer = torch.optim.SGD(
    swag_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4
)
criterion = nn.CrossEntropyLoss()

for epoch in range(20):
    swag_model.train()
    for inputs, labels in mnist_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = swag_model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

    swag_model.update_wrapper(epoch)

    print(f"Эпоха {epoch + 1}")

swag_model.bn_update(mnist_train_loader, device)
swag_model.eval();

## Получение неопределенности и визуализация

In [ ]:
total_id, aleatoric_id, epistemic_id = get_uncertainties(
    swag_model, mnist_id_loader, 20
)
total_ood, aleatoric_ood, epistemic_ood = get_uncertainties(
    swag_model, fmnist_ood_loader, 20
)

In [ ]:
y_true = np.concatenate(
    [np.ones_like(epistemic_ood), np.zeros_like(epistemic_id)]
)
auroc_total = roc_auc_score(y_true, np.concatenate([total_ood, total_id]))

plt.figure(figsize=(6, 5))

sns.kdeplot(
    total_id, fill=True, color="#3498db", label="MNIST (ID)", clip=(0, None)
)
sns.kdeplot(
    total_ood,
    fill=True,
    color="#e74c3c",
    label="Fashion-MNIST (OOD)",
    clip=(0, None),
)

plt.title(f"Полная неопределенность\nAUROC = {auroc_total:.3f}", fontsize=14)
plt.xlabel("Значение энтропии", fontsize=12)
plt.ylabel("Плотность", fontsize=12)
plt.legend()

plt.tight_layout()
plt.show()

График похож на тот, что был у байесовской сети, однако разделение между ID и OOD наблюдениями более выражено. Взглянем на соотношение типов распределенности.

In [ ]:
plt.figure(figsize=(6, 5))

plt.scatter(
    aleatoric_id,
    epistemic_id,
    alpha=0.3,
    color="#3498db",
    label="MNIST (ID)",
    s=15,
)
plt.scatter(
    aleatoric_ood,
    epistemic_ood,
    alpha=0.3,
    color="#e74c3c",
    label="Fashion-MNIST (OOD)",
    s=15,
)

max_val = max(np.max(aleatoric_ood), np.max(epistemic_ood))
plt.plot(
    [0, max_val],
    [0, max_val],
    "k--",
    alpha=0.5,
    label="Эпистемическая = Алеаторной",
)

plt.title("Соотношение типов неопределенности", fontsize=14)
plt.xlabel("Алеаторная", fontsize=13)
plt.ylabel("Эпистемическая", fontsize=13)
plt.legend(fontsize=12)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Как и чисто байесовской сети SWAG-у не удалось корректно разделить типы неопределенностей: алеаторная преобретает для изображений из MNIST и Fashion-MNIST. В другом ноутбуке уже отмечалось, что это может происходить из-за простоты модели (она, в свою очередь, обусловлена простотой задачи).